# Notebook 04 - Optimizacion de Prompts

## Objetivos
- Mejorar precision con especificidad y descomposicion.
- Reducir alucinaciones con restricciones explicitas.
- Comparar prompts con y sin guardrails anti-alucinacion.

## Introduccion
Un LLM puede sonar convincente incluso cuando inventa. Este notebook ensena tecnicas practicas para controlar la salida y reducir alucinaciones.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

In [ ]:
def generar(prompt, max_new_tokens=40, temperature=0.7):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

## 1) Tecnica: especificidad

In [ ]:
vago = 'Resume el informe.'
especifico = (
    'Resume el informe del Q3 en exactamente 3 viñetas. '
    'Cada viñeta maximo 15 palabras. Solo hechos del texto. '
    'Informe: Las ventas del Q3 alcanzaron 2.4M USD con 18% de crecimiento. B2B fue 62%.\n'
    'Resumen:'
)
for nombre, p in [('Vago', vago), ('Especifico', especifico)]:
    print(f'\n=== {nombre} ===')
    print(generar(p, max_new_tokens=40, temperature=0.5))

## 2) Tecnica: descomposicion de tareas

In [ ]:
prompt_descompuesto = (
    'Analiza el mensaje del cliente en 3 pasos:\n'
    'Paso 1: Identifica el problema principal.\n'
    'Paso 2: Clasifica urgencia (alta/media/baja).\n'
    'Paso 3: Sugiere una accion para el equipo de soporte.\n'
    'Mensaje: El sistema está caído y no podemos procesar pagos desde la mañana.\n'
    'Analisis:'
)
print(generar(prompt_descompuesto, max_new_tokens=60, temperature=0.5))

## 3) Anti-alucinacion: con y sin restricciones

In [ ]:
df_docs = pd.read_csv(DATASETS / 'documentos_empresa.csv')
contexto = df_docs[df_docs['titulo'] == 'Reporte Q3 2025'].iloc[0]['contenido']

sin_guardrail = f'Contexto: {contexto}\n¿Cuáles fueron las ventas del Q4?\nRespuesta:'
con_guardrail = (
    f'Contexto: {contexto}\n'
    'Reglas: Responde SOLO con el contexto. Si no está, di No disponible. No inventes.\n'
    '¿Cuáles fueron las ventas del Q4?\n'
    'Respuesta:'
)

comparacion = pd.DataFrame([
    {'version': 'Sin guardrail', 'respuesta': generar(sin_guardrail, max_new_tokens=30, temperature=0.3)},
    {'version': 'Con guardrail', 'respuesta': generar(con_guardrail, max_new_tokens=30, temperature=0.3)},
])
display(comparacion)

## 4) Temperatura: factual vs creativo

In [ ]:
prompt_factual = 'Clasifica como positivo o negativo. Solo etiqueta: El producto se rompió al primer día.'
temps = [0.1, 0.5, 1.0]
filas = []
for t in temps:
    r = generar(prompt_factual, max_new_tokens=8, temperature=t)
    filas.append({'temperatura': t, 'salida': r[-30:]})
display(pd.DataFrame(filas))

## 5) Plantilla de restricciones reutilizable

In [ ]:
RESTRICCIONES_ESTANDAR = """
REGLAS:
- Responde SOLO con el contexto proporcionado
- Si falta informacion, di 'No disponible'
- No inventes cifras, fechas ni nombres
- Maximo 100 palabras
- Tono profesional
"""

def prompt_seguro(contexto, pregunta):
    return f'{RESTRICCIONES_ESTANDAR}\n<contexto>\n{contexto}\n</contexto>\nPregunta: {pregunta}\nRespuesta:'

doc = df_docs[df_docs['titulo'] == 'Plan empresarial'].iloc[0]
p = prompt_seguro(doc['contenido'], '¿Cuántos usuarios incluye el plan Enterprise?')
print(generar(p, max_new_tokens=30, temperature=0.2))

## 6) Checklist de optimizacion

In [ ]:
checklist = pd.DataFrame([
    {'paso': 1, 'accion': 'Definir tarea con verbo claro', 'ejemplo': 'Clasifica / Resume / Traduce'},
    {'paso': 2, 'accion': 'Acotar formato de salida', 'ejemplo': 'JSON / 3 bullets / solo etiqueta'},
    {'paso': 3, 'accion': 'Agregar restricciones anti-alucinacion', 'ejemplo': 'Solo usa el contexto'},
    {'paso': 4, 'accion': 'Elegir tecnica adecuada', 'ejemplo': 'Few-shot para categorias propias'},
    {'paso': 5, 'accion': 'Ajustar temperatura', 'ejemplo': 'Baja para factual, alta para creativo'},
    {'paso': 6, 'accion': 'Probar con 3+ entradas y evaluar', 'ejemplo': 'Tabla comparativa'},
    {'paso': 7, 'accion': 'Iterar basado en errores', 'ejemplo': 'Agregar ejemplos donde falla'},
])
display(checklist)

## Resultados
Comparamos prompts vagos vs especificos, demostramos anti-alucinacion y construimos una plantilla de restricciones reutilizable.

## Conclusiones
La optimizacion es iterativa: diseña, prueba, mide, ajusta. Las restricciones explicitas son la defensa mas simple contra alucinaciones.

## Ejercicios guiados resueltos
**Ejercicio:** Pregunta algo que NO esta en el contexto con prompt_seguro().

**Solucion:**

In [ ]:
p = prompt_seguro(doc['contenido'], '¿Cuál es el nombre del CEO?')
print(generar(p, max_new_tokens=15, temperature=0.2))

## Ejercicios propuestos
1. Crea 5 restricciones para un chatbot medico.
2. Compara temperatura 0.1 vs 1.0 en tarea creativa.
3. Diseña checklist propio para tu equipo.

## Preguntas de reflexion
1. Por que la temperatura baja reduce alucinaciones?
2. Que tareas NUNCA deberian automatizarse sin humano?
3. Como detectarias alucinaciones en produccion?